# Pipeline — Extração PostgreSQL → Landing Zone (MinIO)

Lê tabelas do PostgreSQL via JDBC e grava cada uma como CSV no bucket `landing-zone` do MinIO.

## 1. Configuração

In [4]:
# Conexão PostgreSQL
PG_HOST     = "localhost"
PG_PORT     = "5432"
PG_DB       = "rh"
PG_USER     = "engdados"
PG_PASSWORD = "engdados"
JDBC_URL    = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

# MinIO / S3
MINIO_ENDPOINT   = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
LANDING_BUCKET   = "s3a://landing-zone"

# Tabelas a extrair do PostgreSQL
TABLES = [
    "brands",
    "categories",
    "customers",
    "stores",
    "staffs",
    "products",
    "stocks",
    "orders",
    "order_items",
]

## 2. SparkSession com suporte a S3A (MinIO) e Delta Lake

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("extract_to_landing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.postgresql:postgresql:42.7.3",
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession iniciada — versão: {spark.version}")

SparkSession iniciada — versão: 3.5.0


## 3. Leitura das tabelas via JDBC

In [6]:
from pyspark.sql.functions import current_timestamp, lit

jdbc_props = {
    "user":     PG_USER,
    "password": PG_PASSWORD,
    "driver":   "org.postgresql.Driver",
}

dataframes = {}
for table in TABLES:
    df = (
        spark.read
        .jdbc(url=JDBC_URL, table=table, properties=jdbc_props)
        .withColumn("extracted_at", current_timestamp())
        .withColumn("source_system", lit("postgres"))
    )
    dataframes[table] = df
    print(f"{table}: {df.count()} registros")

brands: 9 registros
categories: 7 registros
customers: 1445 registros
stores: 3 registros
staffs: 10 registros
products: 321 registros
stocks: 939 registros
orders: 1615 registros
order_items: 4722 registros


## 4. Gravação como CSV no bucket landing-zone

In [7]:
for table, df in dataframes.items():
    output_path = f"{LANDING_BUCKET}/{table}"
    (
        df.coalesce(1)          # um único arquivo CSV por tabela
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(output_path)
    )
    print(f"Gravado: {output_path}")

print("\nExtração concluída — dados disponíveis no landing-zone.")

26/05/06 19:44:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/06 19:44:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 19:44:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Gravado: s3a://landing-zone/brands


26/05/06 19:44:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 19:44:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Gravado: s3a://landing-zone/categories


26/05/06 19:44:23 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 19:44:23 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


...

Gravado: s3a://landing-zone/stocks


26/05/06 19:44:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 19:44:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Gravado: s3a://landing-zone/orders


26/05/06 19:44:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 19:44:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Gravado: s3a://landing-zone/order_items

Extração concluída — dados disponíveis no landing-zone.


In [8]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
